In [1]:
!pip install -q pandas scikit-learn joblib

import json
import time
import hashlib
import shutil
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [2]:
DATA_DIR = Path("/content/sample_data")

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"
TEST_PATH = DATA_DIR / "test.csv"

OUTPUT_DIR = Path("/content/classifier_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / "intent_classifier.joblib"
METRICS_PATH = OUTPUT_DIR / "metrics.json"
LABEL_MAPPING_PATH = OUTPUT_DIR / "label_mapping.json"
ROUTER_THRESHOLDS_PATH = OUTPUT_DIR / "router_thresholds.json"
MODEL_CARD_PATH = OUTPUT_DIR / "model_card.md"
TEST_PREDICTIONS_PATH = OUTPUT_DIR / "test_predictions.csv"

EXPECTED_LABELS = [
    "faq",
    "support",
    "sales_or_leads",
    "human_request",
    "spam",
    "other",
]

# Validation-based target for direct routing.
# Messages below threshold go to the agent.
TARGET_ROUTED_ACCURACY = 0.97

print("Train path:", TRAIN_PATH)
print("Val path:", VAL_PATH)
print("Test path:", TEST_PATH)
print("Output dir:", OUTPUT_DIR)

Train path: /content/sample_data/train.csv
Val path: /content/sample_data/val.csv
Test path: /content/sample_data/test.csv
Output dir: /content/classifier_artifacts


In [3]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

def validate_dataset(df: pd.DataFrame, name: str):
    required_columns = {"text", "label"}

    if set(df.columns) != required_columns:
        raise ValueError(
            f"{name} must contain exactly columns {required_columns}. "
            f"Found: {set(df.columns)}"
        )

    if df["text"].isna().any():
        raise ValueError(f"{name} contains empty text values.")

    if df["label"].isna().any():
        raise ValueError(f"{name} contains empty label values.")

    unknown_labels = set(df["label"].unique()) - set(EXPECTED_LABELS)
    if unknown_labels:
        raise ValueError(f"{name} contains unknown labels: {unknown_labels}")

    print(f"{name} is valid.")
    print("Rows:", len(df))
    print(df["label"].value_counts())
    print()

validate_dataset(train_df, "train.csv")
validate_dataset(val_df, "val.csv")
validate_dataset(test_df, "test.csv")

train.csv is valid.
Rows: 9016
label
faq               2100
support           2100
sales_or_leads    2022
human_request     1399
other              945
spam               450
Name: count, dtype: int64

val.csv is valid.
Rows: 1932
label
support           450
faq               450
sales_or_leads    434
human_request     300
other             202
spam               96
Name: count, dtype: int64

test.csv is valid.
Rows: 1932
label
support           450
faq               450
sales_or_leads    433
human_request     300
other             203
spam               96
Name: count, dtype: int64



In [4]:
def clean_text(text: str) -> str:
    text = str(text)
    text = " ".join(text.split())
    return text.strip()

def normalize_for_duplicate_check(text: str) -> str:
    text = clean_text(text)
    return text.lower()

for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].apply(clean_text)
    df["label"] = df["label"].astype(str)
    df["norm_text"] = df["text"].apply(normalize_for_duplicate_check)

print("Example:")
print(train_df[["text", "label", "norm_text"]].head())

Example:
                                                text           label  \
0  i dont know what i need to do to speak to a pe...   human_request   
1                              acquire several items  sales_or_leads   
2                  what trees change color in autumn           other   
3    I want assistance to restore my user access key         support   
4  can you help me sign up to the company newslet...  sales_or_leads   

                                           norm_text  
0  i dont know what i need to do to speak to a pe...  
1                              acquire several items  
2                  what trees change color in autumn  
3    i want assistance to restore my user access key  
4  can you help me sign up to the company newslet...  


In [5]:
# ============================================================
# Clean train set:
# - If exact same normalized text appears multiple times with same label, keep one.
# - If same text appears with conflicting labels, drop it.
# ============================================================

label_counts_per_text = (
    train_df.groupby("norm_text")["label"]
    .nunique()
    .reset_index(name="num_labels")
)

conflicting_train_texts = set(
    label_counts_per_text[label_counts_per_text["num_labels"] > 1]["norm_text"]
)

train_no_conflicts = train_df[
    ~train_df["norm_text"].isin(conflicting_train_texts)
].copy()

train_clean = train_no_conflicts.drop_duplicates(
    subset=["norm_text"],
    keep="first"
).copy()

# ============================================================
# Clean val/test:
# - Remove examples whose exact text appeared in train.
# - Remove exact duplicate texts inside val/test.
# ============================================================

train_clean_texts = set(train_clean["norm_text"])

val_clean = val_df[
    ~val_df["norm_text"].isin(train_clean_texts)
].drop_duplicates(subset=["norm_text"], keep="first").copy()

test_clean = test_df[
    ~test_df["norm_text"].isin(train_clean_texts)
].drop_duplicates(subset=["norm_text"], keep="first").copy()

print("Original train rows:", len(train_df))
print("Train conflicting texts removed:", len(conflicting_train_texts))
print("Clean train rows:", len(train_clean))

print("\nOriginal val rows:", len(val_df))
print("Clean val rows:", len(val_clean))
print("Removed from val:", len(val_df) - len(val_clean))

print("\nOriginal test rows:", len(test_df))
print("Clean test rows:", len(test_clean))
print("Removed from test:", len(test_df) - len(test_clean))

print("\nClean train label counts:")
print(train_clean["label"].value_counts())

print("\nClean val label counts:")
print(val_clean["label"].value_counts())

print("\nClean test label counts:")
print(test_clean["label"].value_counts())

Original train rows: 9016
Train conflicting texts removed: 0
Clean train rows: 8967

Original val rows: 1932
Clean val rows: 1904
Removed from val: 28

Original test rows: 1932
Clean test rows: 1915
Removed from test: 17

Clean train label counts:
label
support           2093
faq               2090
sales_or_leads    2006
human_request     1383
other              945
spam               450
Name: count, dtype: int64

Clean val label counts:
label
support           448
faq               444
sales_or_leads    424
human_request     290
other             202
spam               96
Name: count, dtype: int64

Clean test label counts:
label
faq               446
support           445
sales_or_leads    431
human_request     294
other             203
spam               96
Name: count, dtype: int64


In [6]:
# ============================================================
# Clean train set:
# - If exact same normalized text appears multiple times with same label, keep one.
# - If same text appears with conflicting labels, drop it.
# ============================================================

label_counts_per_text = (
    train_df.groupby("norm_text")["label"]
    .nunique()
    .reset_index(name="num_labels")
)

conflicting_train_texts = set(
    label_counts_per_text[label_counts_per_text["num_labels"] > 1]["norm_text"]
)

train_no_conflicts = train_df[
    ~train_df["norm_text"].isin(conflicting_train_texts)
].copy()

train_clean = train_no_conflicts.drop_duplicates(
    subset=["norm_text"],
    keep="first"
).copy()

# ============================================================
# Clean val/test:
# - Remove examples whose exact text appeared in train.
# - Remove exact duplicate texts inside val/test.
# ============================================================

train_clean_texts = set(train_clean["norm_text"])

val_clean = val_df[
    ~val_df["norm_text"].isin(train_clean_texts)
].drop_duplicates(subset=["norm_text"], keep="first").copy()

test_clean = test_df[
    ~test_df["norm_text"].isin(train_clean_texts)
].drop_duplicates(subset=["norm_text"], keep="first").copy()

print("Original train rows:", len(train_df))
print("Train conflicting texts removed:", len(conflicting_train_texts))
print("Clean train rows:", len(train_clean))

print("\nOriginal val rows:", len(val_df))
print("Clean val rows:", len(val_clean))
print("Removed from val:", len(val_df) - len(val_clean))

print("\nOriginal test rows:", len(test_df))
print("Clean test rows:", len(test_clean))
print("Removed from test:", len(test_df) - len(test_clean))

print("\nClean train label counts:")
print(train_clean["label"].value_counts())

print("\nClean val label counts:")
print(val_clean["label"].value_counts())

print("\nClean test label counts:")
print(test_clean["label"].value_counts())

Original train rows: 9016
Train conflicting texts removed: 0
Clean train rows: 8967

Original val rows: 1932
Clean val rows: 1904
Removed from val: 28

Original test rows: 1932
Clean test rows: 1915
Removed from test: 17

Clean train label counts:
label
support           2093
faq               2090
sales_or_leads    2006
human_request     1383
other              945
spam               450
Name: count, dtype: int64

Clean val label counts:
label
support           448
faq               444
sales_or_leads    424
human_request     290
other             202
spam               96
Name: count, dtype: int64

Clean test label counts:
label
faq               446
support           445
sales_or_leads    431
human_request     294
other             203
spam               96
Name: count, dtype: int64


In [7]:
X_train = train_clean["text"]
y_train = train_clean["label"]

X_val = val_clean["text"]
y_val = val_clean["label"]

X_test = test_clean["text"]
y_test = test_clean["label"]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))

Training rows: 8967
Validation rows: 1904
Test rows: 1915


In [8]:
candidate_models = [
    {
        "name": "dummy_most_frequent",
        "pipeline": Pipeline(
            steps=[
                ("classifier", DummyClassifier(strategy="most_frequent")),
            ]
        ),
    },
    {
        "name": "tfidf_word_1_1_logreg_C1",
        "pipeline": Pipeline(
            steps=[
                (
                    "tfidf",
                    TfidfVectorizer(
                        analyzer="word",
                        ngram_range=(1, 1),
                        max_features=30000,
                        min_df=2,
                        lowercase=True,
                        strip_accents="unicode",
                        sublinear_tf=True,
                    ),
                ),
                (
                    "classifier",
                    LogisticRegression(
                        C=1.0,
                        max_iter=2000,
                        class_weight="balanced",
                        solver="lbfgs",
                        n_jobs=-1,
                        random_state=42,
                    ),
                ),
            ]
        ),
    },
    {
        "name": "tfidf_word_1_2_logreg_C1",
        "pipeline": Pipeline(
            steps=[
                (
                    "tfidf",
                    TfidfVectorizer(
                        analyzer="word",
                        ngram_range=(1, 2),
                        max_features=50000,
                        min_df=2,
                        lowercase=True,
                        strip_accents="unicode",
                        sublinear_tf=True,
                    ),
                ),
                (
                    "classifier",
                    LogisticRegression(
                        C=1.0,
                        max_iter=2000,
                        class_weight="balanced",
                        solver="lbfgs",
                        n_jobs=-1,
                        random_state=42,
                    ),
                ),
            ]
        ),
    },
    {
        "name": "tfidf_word_1_2_logreg_C2",
        "pipeline": Pipeline(
            steps=[
                (
                    "tfidf",
                    TfidfVectorizer(
                        analyzer="word",
                        ngram_range=(1, 2),
                        max_features=50000,
                        min_df=2,
                        lowercase=True,
                        strip_accents="unicode",
                        sublinear_tf=True,
                    ),
                ),
                (
                    "classifier",
                    LogisticRegression(
                        C=2.0,
                        max_iter=2000,
                        class_weight="balanced",
                        solver="lbfgs",
                        n_jobs=-1,
                        random_state=42,
                    ),
                ),
            ]
        ),
    },
]

results = []
best_model = None
best_model_name = None
best_val_macro_f1 = -1

for item in candidate_models:
    name = item["name"]
    model = item["pipeline"]

    print("=" * 80)
    print("Training:", name)

    start_time = time.time()
    model.fit(X_train, y_train)
    train_time_seconds = time.time() - start_time

    val_preds = model.predict(X_val)

    val_accuracy = accuracy_score(y_val, val_preds)
    val_macro_f1 = f1_score(y_val, val_preds, average="macro")
    val_weighted_f1 = f1_score(y_val, val_preds, average="weighted")

    result = {
        "name": name,
        "train_time_seconds": float(train_time_seconds),
        "val_accuracy": float(val_accuracy),
        "val_macro_f1": float(val_macro_f1),
        "val_weighted_f1": float(val_weighted_f1),
    }

    results.append(result)

    print("Validation accuracy:", round(val_accuracy, 4))
    print("Validation macro-F1:", round(val_macro_f1, 4))
    print("Validation weighted-F1:", round(val_weighted_f1, 4))
    print("Training time seconds:", round(train_time_seconds, 2))

    if name != "dummy_most_frequent" and val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        best_model = model
        best_model_name = name

print("\n" + "=" * 80)
print("Selected model:", best_model_name)
print("Best clean validation macro-F1:", round(best_val_macro_f1, 4))

Training: dummy_most_frequent
Validation accuracy: 0.2353
Validation macro-F1: 0.0635
Validation weighted-F1: 0.0896
Training time seconds: 0.01
Training: tfidf_word_1_1_logreg_C1
Validation accuracy: 0.9795
Validation macro-F1: 0.9745
Validation weighted-F1: 0.9795
Training time seconds: 3.74
Training: tfidf_word_1_2_logreg_C1
Validation accuracy: 0.9837
Validation macro-F1: 0.9778
Validation weighted-F1: 0.9838
Training time seconds: 6.31
Training: tfidf_word_1_2_logreg_C2
Validation accuracy: 0.9848
Validation macro-F1: 0.9795
Validation weighted-F1: 0.9848
Training time seconds: 2.09

Selected model: tfidf_word_1_2_logreg_C2
Best clean validation macro-F1: 0.9795


In [9]:
def evaluate_model(model, X, y, split_name: str):
    start_time = time.time()
    preds = model.predict(X)
    predict_time_seconds = time.time() - start_time

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        confidence = np.max(proba, axis=1)
    else:
        confidence = np.ones(len(preds))

    metrics = {
        "split": split_name,
        "rows": int(len(X)),
        "accuracy": float(accuracy_score(y, preds)),
        "macro_f1": float(f1_score(y, preds, average="macro")),
        "weighted_f1": float(f1_score(y, preds, average="weighted")),
        "avg_confidence": float(np.mean(confidence)),
        "median_confidence": float(np.median(confidence)),
        "total_predict_time_seconds": float(predict_time_seconds),
        "avg_latency_ms_per_example": float((predict_time_seconds / len(X)) * 1000),
    }

    report = classification_report(
        y,
        preds,
        labels=EXPECTED_LABELS,
        output_dict=True,
        zero_division=0,
    )

    matrix = confusion_matrix(
        y,
        preds,
        labels=EXPECTED_LABELS,
    )

    print("=" * 80)
    print(f"{split_name.upper()} METRICS")
    print(json.dumps(metrics, indent=2))

    print("\nClassification report:")
    print(
        classification_report(
            y,
            preds,
            labels=EXPECTED_LABELS,
            zero_division=0,
        )
    )

    return {
        "metrics": metrics,
        "classification_report": report,
        "confusion_matrix_labels": EXPECTED_LABELS,
        "confusion_matrix": matrix.tolist(),
        "predictions": preds,
        "confidence": confidence,
    }

val_eval = evaluate_model(best_model, X_val, y_val, "clean_val")
test_eval = evaluate_model(best_model, X_test, y_test, "clean_test")

CLEAN_VAL METRICS
{
  "split": "clean_val",
  "rows": 1904,
  "accuracy": 0.9847689075630253,
  "macro_f1": 0.9794513518597121,
  "weighted_f1": 0.9848100238827702,
  "avg_confidence": 0.900265595043182,
  "median_confidence": 0.9433206335738774,
  "total_predict_time_seconds": 0.11135315895080566,
  "avg_latency_ms_per_example": 0.058483801969960954
}

Classification report:
                precision    recall  f1-score   support

           faq       0.99      0.98      0.99       444
       support       0.98      0.99      0.99       448
sales_or_leads       1.00      0.99      0.99       424
 human_request       1.00      1.00      1.00       290
          spam       0.93      0.99      0.96        96
         other       0.96      0.95      0.95       202

      accuracy                           0.98      1904
     macro avg       0.98      0.98      0.98      1904
  weighted avg       0.98      0.98      0.98      1904

CLEAN_TEST METRICS
{
  "split": "clean_test",
  "rows": 19

In [18]:
# ============================================================
# CREATE CLASSIFIER GOLDEN SET
# Balanced fixed eval set for CI/regression testing
# ============================================================

GOLDEN_SET_PATH = OUTPUT_DIR / "classifier_golden_set.csv"

GOLDEN_EXAMPLES_PER_LABEL = 10
RANDOM_STATE = 42

# Use clean test set if available, otherwise normal test set
if "test_clean" in globals():
    source_df = test_clean.copy()
else:
    source_df = test_df.copy()

# Keep only the columns needed for evaluation
source_df = source_df[["text", "label"]].copy()

golden_parts = []

for label in EXPECTED_LABELS:
    label_df = source_df[source_df["label"] == label]

    if len(label_df) < GOLDEN_EXAMPLES_PER_LABEL:
        print(f"Warning: only {len(label_df)} examples found for label: {label}")
        sample_df = label_df.copy()
    else:
        sample_df = label_df.sample(
            n=GOLDEN_EXAMPLES_PER_LABEL,
            random_state=RANDOM_STATE,
        )

    golden_parts.append(sample_df)

golden_df = pd.concat(golden_parts, ignore_index=True)

# Shuffle final golden set
golden_df = golden_df.sample(
    frac=1,
    random_state=RANDOM_STATE,
).reset_index(drop=True)

golden_df.to_csv(GOLDEN_SET_PATH, index=False)

print("Saved golden set to:")
print(GOLDEN_SET_PATH)

print("\nGolden set shape:", golden_df.shape)
print("\nLabel counts:")
print(golden_df["label"].value_counts())

display(golden_df)

Saved golden set to:
/content/classifier_artifacts/classifier_golden_set.csv

Golden set shape: (60, 2)

Label counts:
label
faq               10
human_request     10
spam              10
support           10
other             10
sales_or_leads    10
Name: count, dtype: int64


,text,label
0,I do not know how I can check your reimburseme...,faq
1,help me seeing how long the damn reimbursement...,faq
2,I want assistance talking with an person,human_request
3,"Free Msg: get Gnarls Barkleys \Crazy\"" rington...",spam
4,i have to change the info on my user account i...,support
5,adjust the contrast of the black and white photos,other
6,where to see at what time customer support ava...,human_request
7,URGENT! Your Mobile number has been awarded wi...,spam
8,I do not know how to change to the standard ac...,support
9,what are my local news stations,other


In [19]:
# ============================================================
# INSPECT GOLDEN SET BY LABEL
# ============================================================

for label in EXPECTED_LABELS:
    print("=" * 80)
    print("LABEL:", label)
    display(golden_df[golden_df["label"] == label][["text", "label"]])

LABEL: faq


,text,label
0,I do not know how I can check your reimburseme...,faq
1,help me seeing how long the damn reimbursement...,faq
13,Ineed assistance to see the withdrawal charges,faq
16,help me seeing in which situations can I ask f...,faq
17,I want to look for the invoice #12588,faq
19,I try to see the rebate status,faq
28,what do I need to do to download my invoice #3...,faq
41,I have to check the reimbursement current stat...,faq
43,help me checking in what cases can I ask for r...,faq
54,check status of order {{Order Number}},faq


LABEL: support


,text,label
4,i have to change the info on my user account i...,support
8,I do not know how to change to the standard ac...,support
15,i dont know what i have to do to update my shi...,support
21,can you help me to ask for damn restitutions o...,support
26,I don't know how I could notify of a problem w...,support
31,changing data on freemium account,support
34,need help switching to my new profile,support
49,I don't know how I can ask for compensations,support
51,I don't know how I could delete the standard a...,support
56,need to report issues with registrations,support


LABEL: sales_or_leads


...

,text,label
2,I want assistance talking with an person,human_request
6,where to see at what time customer support ava...,human_request
12,assistance contacting someone,human_request
22,i have got to see at what time customer suppor...,human_request
29,can you direct to me a human agent?,human_request
35,wanna contact customer assistance,human_request
38,show me at what time I can reach customer assi...,human_request
45,can i speak with an assistant,human_request
46,what fucking hours can I call customer support?,human_request
59,I do not know how I could get in touch with cu...,human_request


LABEL: spam


,text,label
3,"Free Msg: get Gnarls Barkleys \Crazy\"" rington...",spam
7,URGENT! Your Mobile number has been awarded wi...,spam
10,Well done ENGLAND! Get the official poly ringt...,spam
18,Dear Dave this is your final notice to collect...,spam
20,As one of our registered subscribers u can ent...,spam
37,URGENT We are trying to contact you Last weeke...,spam
40,5p 4 alfie Moon's Children in need song on ur ...,spam
44,Please CALL 08712402578 immediately as there i...,spam
48,FreeMsg: Txt: CALL to No: 86888 & claim your r...,spam
55,YOUR CHANCE TO BE ON A REALITY FANTASY SHOW ca...,spam


LABEL: other


,text,label
5,adjust the contrast of the black and white photos,other
9,what are my local news stations,other
11,tell me about personal finance,other
14,how is my driving,other
23,what's the safest pet to be around toddlers,other
25,what are bbc headlinesi,other
33,tell me who aphrodite is,other
36,find articles about the protests in parisi,other
52,am i connected to wifi,other
58,what is the largest state in the us,other


In [20]:
# ============================================================
# EVALUATE MODEL ON GOLDEN SET
# ============================================================

golden_eval_df = pd.read_csv(GOLDEN_SET_PATH)

X_golden = golden_eval_df["text"].astype(str)
y_golden = golden_eval_df["label"].astype(str)

golden_preds = best_model.predict(X_golden)
golden_proba = best_model.predict_proba(X_golden)
golden_confidence = np.max(golden_proba, axis=1)

golden_accuracy = accuracy_score(y_golden, golden_preds)
golden_macro_f1 = f1_score(y_golden, golden_preds, average="macro")
golden_weighted_f1 = f1_score(y_golden, golden_preds, average="weighted")

print("GOLDEN SET METRICS")
print("Accuracy:", round(golden_accuracy, 4))
print("Macro-F1:", round(golden_macro_f1, 4))
print("Weighted-F1:", round(golden_weighted_f1, 4))
print("Average confidence:", round(float(np.mean(golden_confidence)), 4))

print("\nClassification report:")
print(
    classification_report(
        y_golden,
        golden_preds,
        labels=EXPECTED_LABELS,
        zero_division=0,
    )
)

golden_results_df = golden_eval_df.copy()
golden_results_df["predicted_label"] = golden_preds
golden_results_df["confidence"] = golden_confidence
golden_results_df["correct"] = golden_results_df["label"] == golden_results_df["predicted_label"]

display(golden_results_df)

GOLDEN SET METRICS
Accuracy: 1.0
Macro-F1: 1.0
Weighted-F1: 1.0
Average confidence: 0.9188

Classification report:
                precision    recall  f1-score   support

           faq       1.00      1.00      1.00        10
       support       1.00      1.00      1.00        10
sales_or_leads       1.00      1.00      1.00        10
 human_request       1.00      1.00      1.00        10
          spam       1.00      1.00      1.00        10
         other       1.00      1.00      1.00        10

      accuracy                           1.00        60
     macro avg       1.00      1.00      1.00        60
  weighted avg       1.00      1.00      1.00        60



,text,label,predicted_label,confidence,correct
0,I do not know how I can check your reimburseme...,faq,faq,0.840656,True
1,help me seeing how long the damn reimbursement...,faq,faq,0.877064,True
2,I want assistance talking with an person,human_request,human_request,0.972086,True
3,"Free Msg: get Gnarls Barkleys \Crazy\"" rington...",spam,spam,0.981830,True
4,i have to change the info on my user account i...,support,support,0.969810,True
5,adjust the contrast of the black and white photos,other,other,0.948410,True
6,where to see at what time customer support ava...,human_request,human_request,0.973430,True
7,URGENT! Your Mobile number has been awarded wi...,spam,spam,0.963277,True
8,I do not know how to change to the standard ac...,support,support,0.945324,True
9,what are my local news stations,other,other,0.949004,True


In [21]:
# ============================================================
# SAVE GOLDEN SET PREDICTIONS
# ============================================================

GOLDEN_RESULTS_PATH = OUTPUT_DIR / "classifier_golden_results.csv"

golden_results_df.to_csv(GOLDEN_RESULTS_PATH, index=False)

print("Saved golden results to:")
print(GOLDEN_RESULTS_PATH)

Saved golden results to:
/content/classifier_artifacts/classifier_golden_results.csv


In [10]:
# ============================================================
# NORMAL TESTING CELL — No threshold, no routing
# ============================================================

def predict_message(text: str):
    cleaned = clean_text(text)

    predicted_label = best_model.predict([cleaned])[0]
    probabilities = best_model.predict_proba([cleaned])[0]
    confidence = float(np.max(probabilities))

    return {
        "label": predicted_label,
        "confidence": round(confidence, 4),
    }


manual_tests = [
    "What are your opening hours?",
    "I want to talk to a real person please",
    "Can someone contact me about pricing?",
    "You won a free prize click this link now",
    "My order is not working and I need help",
    "hello there",
    "help me check what hours i can reach customer support",
    "i do not know what i need to do to speak with an assistant",
]

for msg in manual_tests:
    print(msg)
    print(predict_message(msg))
    print()

What are your opening hours?
{'label': 'other', 'confidence': 0.7085}

I want to talk to a real person please
{'label': 'human_request', 'confidence': 0.8933}

Can someone contact me about pricing?
{'label': 'human_request', 'confidence': 0.8158}

You won a free prize click this link now
{'label': 'spam', 'confidence': 0.9869}

My order is not working and I need help
{'label': 'other', 'confidence': 0.2908}

hello there
{'label': 'other', 'confidence': 0.3796}

help me check what hours i can reach customer support
{'label': 'human_request', 'confidence': 0.9909}

i do not know what i need to do to speak with an assistant
{'label': 'human_request', 'confidence': 0.9816}



In [11]:
# ============================================================
# Inspect dataset examples for confusing keywords
# ============================================================

def search_dataset(keyword, n=20):
    keyword = keyword.lower()

    combined = pd.concat(
        [
            train_clean.assign(split="train"),
            val_clean.assign(split="val"),
            test_clean.assign(split="test"),
        ],
        ignore_index=True,
    )

    matches = combined[
        combined["text"].str.lower().str.contains(keyword, na=False)
    ][["split", "text", "label"]]

    print(f"Matches for: {keyword}")
    print("Count:", len(matches))
    display(matches.head(n))

search_dataset("pricing")
search_dataset("price")
search_dataset("hours")
search_dataset("support")
search_dataset("order")
search_dataset("contact")

Matches for: pricing
Count: 1


,split,text,label
3500,train,will amazon ship an order even if there was a ...,other


Matches for: price
Count: 33


,split,text,label
893,train,google the price of skydiving in florida,other
1950,train,can you tell me how to compute price per ounce,other
2154,train,4mths half price Orange line rental & latest c...,spam
2180,train,Do you want a new video handset? 750 anytime a...,spam
2205,train,Double your mins & txts on Orange or 1/2 price...,spam
2288,train,what are apartment prices now,other
2436,train,look up prices for parking at the airport over...,other
2945,train,give me prices on a copy of the mla grammar ha...,other
2991,train,FREE camera phones with linerental from 4.49/m...,spam
3809,train,what's the best price on a gray xl ll bean men...,other


Matches for: hours
Count: 270


...

,split,text,label
8,train,assistance seeing at what time I can call cust...,human_request
41,train,I do not know how I could contact fucking cust...,human_request
77,train,I need to talk with bloody customer support,human_request
110,train,uhave a free of charge number to talk with cus...,human_request
163,train,check at what time i can reachy customer support,human_request
183,train,I try to see what hours I can call customer su...,human_request
291,train,I don't know what I need to do to speak to cus...,human_request
347,train,I want support trying to correct my delivery a...,support
369,train,help me to see at what time customer support a...,human_request
424,train,wanna see what hours customer support availabl...,human_request


Matches for: order
Count: 903


,split,text,label
12,train,"I missed an item in purchase {{Order Number}},...",support
13,train,i dont want this artilce cancel order {{Order ...,support
53,train,need to eddit order {{Order Number}},support
85,train,add item to order,support
113,train,I have got to add an article to order {{Order ...,support
115,train,i want help ordering some articles,sales_or_leads
125,train,i have to change an item of order {{Order Numb...,support
142,train,i nee help to cancel order {{Order Number}},support
162,train,I try to see the ETA of the purchase {{Order N...,faq
191,train,need assistance ordering some goddamn items,sales_or_leads


Matches for: contact
Count: 384


,split,text,label
41,train,I do not know how I could contact fucking cust...,human_request
120,train,contacting agent,human_request
139,train,how do I see what hours I can contact customer...,human_request
161,train,how do I see at what time I can contact custom...,human_request
236,train,i dont know how to contact a human agent,human_request
294,train,how can I contact a human agent?,human_request
323,train,show me all contacts,other
325,train,how do i contact someone,human_request
363,train,I am trying to contact a agent,human_request
415,train,what do I need to do to contact an operator?,human_request


In [12]:
# ============================================================
# CELL 12 — Save model artifact and label metadata
# No threshold, no router logic
# ============================================================

joblib.dump(best_model, MODEL_PATH)

labels_from_model = best_model.named_steps["classifier"].classes_.tolist()

label_mapping = {
    "labels": labels_from_model,
    "expected_labels": EXPECTED_LABELS,
    "response_format": {
        "label": "string",
        "confidence": "float",
    },
}

with open(LABEL_MAPPING_PATH, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2)

print("Saved model:")
print(MODEL_PATH)

print("\nSaved label mapping:")
print(LABEL_MAPPING_PATH)

print("\nLabels from model:")
print(labels_from_model)

Saved model:
/content/classifier_artifacts/intent_classifier.joblib

Saved label mapping:
/content/classifier_artifacts/label_mapping.json

Labels from model:
['faq', 'human_request', 'other', 'sales_or_leads', 'spam', 'support']


In [13]:
# ============================================================
# CELL 13 — Save metrics.json
# No threshold, no router metrics
# ============================================================

all_metrics = {
    "task": "Concierge visitor message intent classification",
    "model_type": "TF-IDF word n-grams + Logistic Regression",
    "selected_model": best_model_name,
    "dataset": {
        "original_train_rows": int(len(train_df)),
        "original_val_rows": int(len(val_df)),
        "original_test_rows": int(len(test_df)),
        "clean_train_rows": int(len(train_clean)) if "train_clean" in globals() else int(len(train_df)),
        "clean_val_rows": int(len(val_clean)) if "val_clean" in globals() else int(len(val_df)),
        "clean_test_rows": int(len(test_clean)) if "test_clean" in globals() else int(len(test_df)),
        "columns": ["text", "label"],
        "labels": EXPECTED_LABELS,
        "train_conflicting_texts_removed": int(len(conflicting_train_texts)) if "conflicting_train_texts" in globals() else 0,
    },
    "candidate_results": results,
    "validation": {
        "metrics": val_eval["metrics"],
        "classification_report": val_eval["classification_report"],
        "confusion_matrix_labels": val_eval["confusion_matrix_labels"],
        "confusion_matrix": val_eval["confusion_matrix"],
    },
    "test": {
        "metrics": test_eval["metrics"],
        "classification_report": test_eval["classification_report"],
        "confusion_matrix_labels": test_eval["confusion_matrix_labels"],
        "confusion_matrix": test_eval["confusion_matrix"],
    },
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2)

print("Saved metrics:")
print(METRICS_PATH)

Saved metrics:
/content/classifier_artifacts/metrics.json


In [14]:
# ============================================================
# CELL 15 — Compute SHA-256 hash for the model artifact
# ============================================================

def sha256_file(path: Path) -> str:
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

model_sha256 = sha256_file(MODEL_PATH)

print("Model artifact SHA-256:")
print(model_sha256)

Model artifact SHA-256:
cd9df32928a567d39a8b5e0246112e1dc6d80a6a215cef336b58e99e5bbae7f0


In [15]:
# ============================================================
# CELL 16 — Create model_card.md
# No threshold, no router metrics
# ============================================================

clean_train_rows = int(len(train_clean)) if "train_clean" in globals() else int(len(train_df))
clean_val_rows = int(len(val_clean)) if "val_clean" in globals() else int(len(val_df))
clean_test_rows = int(len(test_clean)) if "test_clean" in globals() else int(len(test_df))
removed_conflicts = int(len(conflicting_train_texts)) if "conflicting_train_texts" in globals() else 0

model_card = f"""# Model Card: Concierge Intent Classifier

## Task Description

This model classifies incoming visitor messages into one of six Concierge router categories:

- faq
- support
- sales_or_leads
- human_request
- spam
- other

The model returns a predicted label and a confidence score. The classifier is intended to support the Concierge routing layer by giving a cheap first-pass intent prediction before the main system decides what to do next.

## Dataset

The original dataset was split into:

- train.csv: {len(train_df):,} rows
- val.csv: {len(val_df):,} rows
- test.csv: {len(test_df):,} rows

Each file has two columns:

- text
- label

## Duplicate and Leakage Handling

To make evaluation more honest:

- duplicated training messages were deduplicated
- training texts with conflicting labels were removed
- validation/test messages that appeared exactly in training were removed
- duplicate messages inside validation/test were also removed

Clean evaluation sizes:

- clean train: {clean_train_rows:,} rows
- clean validation: {clean_val_rows:,} rows
- clean test: {clean_test_rows:,} rows

Training texts with conflicting labels removed: {removed_conflicts}

## Model

Selected model:

- {best_model_name}
- TF-IDF word features
- Logistic Regression
- exported as joblib

## Output Format

```json
{{
  "label": "faq",
  "confidence": 0.91
}}
Validation Results
Accuracy: {val_eval["metrics"]["accuracy"]:.4f}
Macro-F1: {val_eval["metrics"]["macro_f1"]:.4f}
Weighted-F1: {val_eval["metrics"]["weighted_f1"]:.4f}
Average confidence: {val_eval["metrics"]["avg_confidence"]:.4f}
Average latency per example: {val_eval["metrics"]["avg_latency_ms_per_example"]:.4f} ms
Test Results
Accuracy: {test_eval["metrics"]["accuracy"]:.4f}
Macro-F1: {test_eval["metrics"]["macro_f1"]:.4f}
Weighted-F1: {test_eval["metrics"]["weighted_f1"]:.4f}
Average confidence: {test_eval["metrics"]["avg_confidence"]:.4f}
Average latency per example: {test_eval["metrics"]["avg_latency_ms_per_example"]:.4f} ms
Deployment Choice

This model is suitable for the first shipped classifier because it is lightweight, fast, and can run in the modelserver with scikit-learn and joblib only. It does not require torch or transformers in the serving container.

Limitations

Manual testing showed that some real Concierge-style messages can be ambiguous. For example, a message like "Can someone contact me about pricing?" can look like both a sales lead and a human request. This is a label-definition issue, not only a model issue.

The dataset also appears cleaner and more template-like than real website chat traffic, so production behavior should be monitored using real widget messages over time.

Artifact
Artifact file: intent_classifier.joblib
SHA-256: {model_sha256}
"""

with open(MODEL_CARD_PATH, "w", encoding="utf-8") as f:
 f.write(model_card)

print("Saved model card:")
print(MODEL_CARD_PATH)

print(model_card)



Saved model card:
/content/classifier_artifacts/model_card.md
# Model Card: Concierge Intent Classifier

## Task Description

This model classifies incoming visitor messages into one of six Concierge router categories:

- faq
- support
- sales_or_leads
- human_request
- spam
- other

The model returns a predicted label and a confidence score. The classifier is intended to support the Concierge routing layer by giving a cheap first-pass intent prediction before the main system decides what to do next.

## Dataset

The original dataset was split into:

- train.csv: 9,016 rows
- val.csv: 1,932 rows
- test.csv: 1,932 rows

Each file has two columns:

- text
- label

## Duplicate and Leakage Handling

To make evaluation more honest:

- duplicated training messages were deduplicated
- training texts with conflicting labels were removed
- validation/test messages that appeared exactly in training were removed
- duplicate messages inside validation/test were also removed

Clean evaluation size

In [16]:
ZIP_PATH = "/content/concierge_intent_classifier_artifacts"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    OUTPUT_DIR,
)

print("Created zip file:")
print(ZIP_PATH + ".zip")

Created zip file:
/content/concierge_intent_classifier_artifacts.zip


In [17]:
# ============================================================
# CELL 18 — Download artifacts
# ============================================================

from google.colab import files

files.download(ZIP_PATH + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>